In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [ ]:
import os
import sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

sys.path.append("../02-encoding")

In [ ]:
### Shortcut for package import
from pkgimp import *

from nb2p import (
    codeop,
    fileop,
    token,
    database,
    dgraph,
    coderepr,
    config,
    npop,
    jsonencode,
    astparse,
    notebook
)
from nb2p.notebook import Notebook
from nb2p.stmodel.dnn import NB2PDecoder, Transformer, BiLSTM
from nb2p import database, config, astparse
from dfgtree import DFGTree, preprocess

In [ ]:
DATASET_NAME = "distilkaggle"

In [ ]:
DIRS = config.dirs(dataset_name=DATASET_NAME)
DIRS.makedirs()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
MAX_LENGTH = 256
MODEL_SETUP = f"astn4_{MAX_LENGTH}"
MODEL_SETUP

## Loading Encoding Model

In [ ]:
# Load model directly
from tokenizers import Tokenizer
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig, RobertaForSequenceClassification
# from model import Model
from compressor.model import Model

In [ ]:
torch.set_num_threads(1)
torch.get_num_threads()

In [ ]:
def get_xs_model(model_dir: str, size: int):
    config = RobertaConfig.from_pretrained("microsoft/graphcodebert-base")
    config.num_attention_heads = 8
    config.hidden_size = 96
    config.intermediate_size = 64
    config.vocab_size = 1000
    config.num_hidden_layers = 12
    config.hidden_dropout_prob = 0.2

    tokenizer_path = os.path.join("../02-encoding/compressor", "BPE" + "_" + str(config.vocab_size) + ".json")
    tokenizer = Tokenizer.from_file(tokenizer_path)

    model = Model(RobertaForSequenceClassification(config=config), config, tokenizer)

    model_dir = os.path.join(model_dir, str(size), "model.bin")
    model.load_state_dict(torch.load(model_dir))

    return model, tokenizer

In [ ]:
DEVICE = torch.device("cuda")

model, tokenizer = get_xs_model('../02-encoding/compressor/GraphCodeBERT/clone_detection/checkpoint', 3)
model = model.to(DEVICE)
model.eval()
model

In [ ]:
from dfg import CodeEncodingBuilder

encoding_builder = CodeEncodingBuilder(tokenizer, model)
encoding_builder

In [ ]:
parser, lang = astparse.parser()

db, client = database.connect(dataset_name=DATASET_NAME, verbose=True)

## List Test Data

In [ ]:
# sample test data

sample_nb_ids = [
    str(r["_id"])
    for r in db.notebook.find(
        {"n_ast_children_of_segments": {"$lt": 256}},
        {"n_ast_children_of_segments": 1},
    )
]
print(len(sample_nb_ids), sample_nb_ids[0])

In [ ]:
import logging
logging.getLogger().setLevel(logging.WARN)

## ST Models

In [ ]:
from online import build_shallow_input

def inference(
    model_abbr: str,
    func_do_inference: Callable[[Sequence, Sequence], Sequence],
    detach_after_repr: bool,
    use_mask: bool,
):
    DATA_X = sample_nb_ids

    y_predicts = []

    for id in tqdm(DATA_X, total=len(DATA_X)):
        try:
            nb_data = database.get_notebook(db, ObjectId(id), include_segments=True)
            n_ast_children = nb_data["n_ast_children_of_segments"]

            nb = Notebook.from_db_result(nb_data)
        except:
            y_predicts.append({"notebook_id": id, "segment_ends_pred": []})
            continue

        try:
            start_time = time.time()
            
            pr = preprocess(nb, parser, lang)
            tree = DFGTree(input=pr, builder=encoding_builder).build()
            
            repr_time = time.time() - start_time
        except Exception as e:
            print(f"WARN {id} embed error. reason: {e}")
            continue

        try:
            encoding = build_shallow_input(tree, MAX_LENGTH)
            X = encoding['x']
            X = torch.from_numpy(np.array(npop.pad(X, MAX_LENGTH)[np.newaxis, ...])).to(
                device
            )
        except Exception as e:
            print(f"WARN {id} embed error. reason: {e}")
            y_predicts.append({"notebook_id": id, "segment_ends_pred": []})
            continue

        if detach_after_repr:
            X = X.detach().cpu().numpy()
            X = npop.to_1d(X)

        start_time = time.time()

        if use_mask:
            y_mask = npop.mask(n_ast_children, MAX_LENGTH)[np.newaxis, ...]
            y_pred_eval = func_do_inference(X, y_mask)
        else:
            y_pred_eval = func_do_inference(X, [])

        st_time = time.time() - start_time

        y_pred: np.ndarray = npop.multi_label_binary(y_pred_eval)
        y_pred = y_pred[0][:n_ast_children]
        y_pred[-1] = 1

        ast_ids = np.where(y_pred == 1)[0]

        y_pred: np.ndarray = npop.indices_to_binary(ast_ids, n_ast_children)

        y_predicts.append({"notebook_id": id, "segment_ends_pred": y_pred})

        # db.profile.replace_one(
        #     {
        #         "notebook_id": ObjectId(nb.id),
        #         "dataset": DATASET_NAME,
        #         "method": "st",
        #         "model": model_abbr,
        #     },
        #     {
        #         "notebook_id": ObjectId(nb.id),
        #         "dataset": DATASET_NAME,
        #         "method": "st",
        #         "model": model_abbr,
        #         "n_ast_children": n_ast_children,
        #         "repr_time": repr_time,
        #         "st_time": st_time,
        #     },
        #     upsert=True,
        # )
        # print({
        #     "notebook_id": ObjectId(nb.id),
        #     "dataset": DATASET_NAME,
        #     "method": "st",
        #     "model": model_abbr,
        #     "n_ast_children": n_ast_children,
        #     "repr_time": repr_time,
        #     "st_time": st_time,
        # })

    return y_predicts


def save_result(model_abbr: str, y_predicts: Sequence):
    with open(DIRS.base / f"pred_segment_ends-{model_abbr}-{SETUP}.json", "w") as f:
        json.dump(y_predicts, f, cls=jsonencode.NpEncoder)

### Decision Tree

In [ ]:
MODEL_ABBR = "dtree"
MAX_DEPTH = 64
MIN_SAMPLE_SPLIT = 0.001

In [ ]:
st_model = fileop.read_joblib(
    DIRS.model
    / f"{MODEL_SETUP}-{MODEL_ABBR}-depth_{MAX_DEPTH}-minsplit_{MIN_SAMPLE_SPLIT}.joblib"
)
st_model.verbose = 0

In [ ]:
y_predicts = inference(
    MODEL_ABBR, lambda X, _: st_model.predict(X), detach_after_repr=True, use_mask=False
)

In [ ]:
# save_result(MODEL_ABBR, y_predicts)

### Random Forest

In [ ]:
MODEL_ABBR = "rforest"
MAX_DEPTH = 64
NUM_ESTIMATORS = 100
MIN_SAMPLE_SPLIT = 0.001

In [ ]:
st_model = fileop.read_joblib(
    DIRS.model
    / f"{MODEL_SETUP}-{MODEL_ABBR}-n_{NUM_ESTIMATORS}-depth_{MAX_DEPTH}-minsplit_{MIN_SAMPLE_SPLIT}.joblib"
)
st_model.verbose = 0

In [ ]:
y_predicts = inference(
    MODEL_ABBR, lambda X, _: st_model.predict(X), detach_after_repr=True, use_mask=False
)

print(y_predicts[0])

In [ ]:
# save_result(MODEL_ABBR, y_predicts)

### XGBoost

In [ ]:
MODEL_ABBR = "xgboost"
MAX_DEPTH = 8
NUM_ESTIMATORS = 100
MIN_SAMPLE_SPLIT = 0.001

In [ ]:
from nb2p.stmodel.xgboost import XGBoostModelCPU
import xgboost as xgb

st_model = XGBoostModelCPU(
    params={
        "random_state": 42,
        "num_boost_round": NUM_ESTIMATORS,
        "n_jobs": NUM_ESTIMATORS,
        "verbosity": 3,
        "max_depth": MAX_DEPTH,
        "min_child_weight": int(MIN_SAMPLE_SPLIT * 48509),
    },
    log_path=DIRS.log
    / f"{MODEL_SETUP}-{MODEL_ABBR}-depth_{MAX_DEPTH}-minsplit_{MIN_SAMPLE_SPLIT}",
    model_path=DIRS.model
    / f"{MODEL_SETUP}-{MODEL_ABBR}-depth_{MAX_DEPTH}-minsplit_{MIN_SAMPLE_SPLIT}.joblib",
)

st_model.load()

In [ ]:
y_predicts = inference(
    MODEL_ABBR,
    lambda X, _: st_model.model.predict(xgb.QuantileDMatrix(np.array(npop.to_1d(X)))),
    detach_after_repr=True,
    use_mask=False,
)

print(y_predicts[0])

In [ ]:
# save_result(MODEL_ABBR, y_predicts)

### Transformer

In [ ]:
MODEL_ABBR = "transformer"
MODEL_NAME = "transformer"

num_heads = 8
hidden_size = 512
num_layers = 4

In [ ]:
# Instantiate the model
st_model = Transformer(
    input_size=MAX_LENGTH,
    hidden_size=hidden_size,
    num_heads=num_heads,
    num_layers=num_layers,
).to(device)

MODEL_PATH = (
    DIRS.log / f"{MODEL_SETUP}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}-e50.pt"
)
print(MODEL_PATH)

st_model.load_state_dict(torch.load(MODEL_PATH))
st_model.eval()
print("Model loaded")

In [ ]:
y_predicts = inference(
    MODEL_ABBR,
    lambda X, mask: st_model(X, torch.tensor(mask).to(device)).detach().cpu(),
    detach_after_repr=False,
    use_mask=True,
)

print(y_predicts[0])

In [ ]:
# save_result(MODEL_ABBR, y_predicts)

### NB2P w/o DTE

In [ ]:
MODEL_ABBR = "bilstm"
MODEL_NAME = "bilstm"

hidden_size = 512
num_layers = 4

In [ ]:
# Instantiate the model
st_model = BiLSTM(
    input_size=MAX_LENGTH, hidden_size=hidden_size, num_layers=num_layers
).to(device)

MODEL_PATH = (
    DIRS.log / f"{MODEL_SETUP}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}-e50.pt"
)
print(MODEL_PATH)

st_model.load_state_dict(torch.load(MODEL_PATH))
st_model.eval()
print("Model loaded")

In [ ]:
y_predicts = inference(
    MODEL_ABBR,
    lambda X, mask: st_model(X, torch.tensor(mask).to(device)).detach().cpu(),
    detach_after_repr=False,
    use_mask=True,
)

print(y_predicts[0])

In [ ]:
save_result(MODEL_ABBR, y_predicts)

### NB2P

In [ ]:
from online import build_deep_input

def inference(
    model_abbr: str,
    func_do_inference: Callable[[Sequence, Sequence], Sequence],
    detach_after_repr: bool,
    use_mask: bool,
):
    DATA_X = sample_nb_ids

    y_predicts = []

    for id in tqdm(DATA_X, total=len(DATA_X)):
        try:
            nb_data = database.get_notebook(db, ObjectId(id), include_segments=True)
            n_ast_children = nb_data["n_ast_children_of_segments"]

            nb = Notebook.from_db_result(nb_data)
        except:
            y_predicts.append({"notebook_id": id, "segment_ends_pred": []})
            continue

        try:
            start_time = time.time()
            
            pr = preprocess(nb, parser, lang)
            tree = DFGTree(input=pr, builder=encoding_builder).build()
            
            repr_time = time.time() - start_time
        except Exception as e:
            print(f"WARN {id} embed error. reason: {e}")
            continue

        try:
            encoding = build_deep_input(tree, MAX_LENGTH, device)
            X = [encoding['x']]
            # X = torch.from_numpy(np.array(npop.pad(X, MAX_LENGTH)[np.newaxis, ...])).to(
            #     device
            # )
        except Exception as e:
            print(f"WARN {id} embed error. reason: {e}")
            y_predicts.append({"notebook_id": id, "segment_ends_pred": []})
            continue

        if detach_after_repr:
            X = X.detach().cpu().numpy()
            X = npop.to_1d(X)

        start_time = time.time()

        if use_mask:
            y_mask = npop.mask(n_ast_children, MAX_LENGTH)[np.newaxis, ...]
            y_pred_eval = func_do_inference(X, y_mask)
        else:
            y_pred_eval = func_do_inference(X, [])

        st_time = time.time() - start_time

        y_pred: np.ndarray = npop.multi_label_binary(y_pred_eval)
        y_pred = y_pred[0][:n_ast_children]
        y_pred[-1] = 1

        ast_ids = np.where(y_pred == 1)[0]

        y_pred: np.ndarray = npop.indices_to_binary(ast_ids, n_ast_children)

        y_predicts.append({"notebook_id": id, "segment_ends_pred": y_pred})

        db.profile.replace_one(
            {
                "notebook_id": ObjectId(nb.id),
                "dataset": DATASET_NAME,
                "method": "st",
                "model": model_abbr,
            },
            {
                "notebook_id": ObjectId(nb.id),
                "dataset": DATASET_NAME,
                "method": "st",
                "model": model_abbr,
                "n_ast_children": n_ast_children,
                "repr_time": repr_time,
                "st_time": st_time,
            },
            upsert=True,
        )
        # print({
        #     "notebook_id": ObjectId(nb.id),
        #     "dataset": DATASET_NAME,
        #     "method": "st",
        #     "model": model_abbr,
        #     "n_ast_children": n_ast_children,
        #     "repr_time": repr_time,
        #     "st_time": st_time,
        # })

    return y_predicts

In [ ]:
from nb2p.stmodel.dnn import NB2PDecoder, Trainer, DNNInput

MODEL_ABBR = "nb2p"
MODEL_NAME = "nb2pdecoder"

hidden_size = 512
num_layers = 3
num_internal_layers = 4
epoch = 50

model = NB2PDecoder(
    input_size=MAX_LENGTH, 
    hidden_size=hidden_size,
    num_layers=num_layers,
    num_internal_layers=num_internal_layers,
    epsilon_scale=1.0,
    device=device,
    setup='bce',
).to(device)

MODEL_PATH = (
    DIRS.log
    / f"{MODEL_SETUP}-{MODEL_NAME}_bf_ff{hidden_size}_l{num_layers}_il{num_internal_layers}-e{epoch}.pt"
)
print(MODEL_PATH)

model.load_state_dict(torch.load(MODEL_PATH))
model.eval()
print("Model loaded")

In [ ]:
y_predicts = inference(
    MODEL_ABBR,
    lambda X, mask: model(X, torch.tensor(mask).to(device)).detach().cpu(),
    detach_after_repr=False,
    use_mask=True,
)

print(y_predicts[0])